In [18]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [19]:
data = pd.read_csv('C:\\Study content\\ML\\DATASETS\\IMDB-Dataset.csv')
data.columns = ['text', 'label']
data['text'] = data['text'].str.lower().str.split()

le = LabelEncoder()
data['label'] = le.fit_transform(data['label'])

train_data,test_data = train_test_split(data, test_size= 0.2 , random_state=42)

vocab = {word for phrase in data['text'] for word in phrase}
word_to_idx = {word: idx for idx, word in enumerate(vocab, start=1)}

max_length = data['text'].str.len().max()
def encode_and_pad(text):
    encoded = [word_to_idx[word] for word in text]
    return encoded + [0] * (max_length - len(encoded))

train_data['text'] = train_data['text'].apply(encode_and_pad)
test_data['text'] = test_data['text'].apply(encode_and_pad)

In [20]:
class SentimentDataset(Dataset):
    def __init__(self, data):
        self.texts = data['text'].values
        self.labels = data['label'].values
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        return torch.tensor(text, dtype=torch.long), torch.tensor(label, dtype=torch.long)

train_dataset = SentimentDataset(train_data)
test_dataset = SentimentDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [21]:
class SentimentRnn(nn.Module):
    def __init__(self,vocab_size,embed_size,hidden_size,output_size):
        super(SentimentRnn,self).__init__()
        self.embedding = nn.Embedding(vocab_size,embed_size)
        self.rnn = nn.RNN(embed_size,hidden_size,batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.embedding(x)
        h0 = torch.zeros(1, x.size(0), hidden_size).to(x.device)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out   

vocab_size = len(vocab) +1
embed_size = 128
hidden_size= 128
output_size = 2
model = SentimentRnn(vocab_size,embed_size,hidden_size,output_size)
    


In [ ]:
loss_func  = nn.CrossEntropyLoss()
optimizers = optim.Adam(model.parameters(),lr=0.001)

epochs = 10
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for texts,labels in train_loader:
        output = model(texts)
        loss = loss_func(output,labels)
        optimizers.zero_grad()
        loss.backward()
        optimizers.step()

        epoch_loss += loss.item()
    
    print(f'Epoch [{epoch+1}/{epochs}],Loss: {epoch_loss/len(train_loader):4f}')